# 1. Import Necessary Packages

In [ ]:
# Uncomment and run the following line if required packages are not installed
# !pip install pandas numpy scipy statsmodels matplotlib seaborn openpyxl

# Core data handling
import pandas as pd        # Tabular data structures (DataFrame), I/O, groupby/merge, etc.
import numpy as np         # Numerical computing: arrays, vectorization, stats helpers

# Plotting
import matplotlib.pyplot as plt  # Base plotting (figures, axes, annotations, savefig)
import seaborn as sns            # Statistical/beautiful plots built on matplotlib (themes, high-level charts)

# File & paths
import os                        # File system ops (paths, mkdir, exists), environment variables

# Statistics / tests
from scipy.stats import mannwhitneyu
from scipy import stats 

# Others
from matplotlib.backends.backend_pdf import PdfPages  # Save multiple figures into a single PDF
import matplotlib.image as mpimg                      # Read/display images (e.g., when assembling figure grids)

import matplotlib as mpl


# 2. Set up

## 2.1 Set Up Working Path

In [ ]:
from pathlib import Path

# Change this to the location of the downloaded ReplicationPackage folder.
PROJECT_PATH = Path(r"C:\Your\Path\To\ReplicationPackage")

DATA_PATH = PROJECT_PATH / "Data"
RESULT_PATH = PROJECT_PATH / "Result"
INPUT_DATA_FILE = DATA_PATH / "GNPD-AllFoodDrink_Claim_NPMScore_2015_2024_synthetic.xlsx"

if not INPUT_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Input data file not found: {INPUT_DATA_FILE}\n"
        "Update PROJECT_PATH to the location of your ReplicationPackage folder."
    )

RESULT_PATH.mkdir(parents=True, exist_ok=True)

print("Project path:", PROJECT_PATH)
print("Input data:", INPUT_DATA_FILE)
print("Result path:", RESULT_PATH)


## 2.2 Set Up Formatting

In [ ]:
# Apply Nature-style formatting globally
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'figure.dpi': 600,
})

# Confirm update
print("Matplotlib parameters updated to Nature Food journal style.")

# Set font sizes for readability
plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 14,
    'legend.fontsize': 10,
    'figure.titlesize': 8
})


## 2.3 Load Dataset

In [ ]:
df = pd.read_excel(INPUT_DATA_FILE, engine="openpyxl")
print(f"Loaded {len(df):,} synthetic product records.")


# 3. Results

## 3.1 Tables

### Table 1. The Healthfulness of New Products by Category

In [ ]:
# If needed, install python-docx once:
# !pip install python-docx

from docx import Document
from docx.shared import Inches
from docx.enum.section import WD_ORIENT
import os
import pandas as pd

# ------------------------------------------------------------
# 1. Construct Less healthy (%) under NPM
#    Food: np_score >= 4
#    Beverages: np_score >= 1
# ------------------------------------------------------------
df = df.copy()

df["less_healthy_npm"] = pd.NA
df.loc[df["NewCategory"] == "Beverages", "less_healthy_npm"] = (
    pd.to_numeric(df.loc[df["NewCategory"] == "Beverages", "np_score"], errors="coerce") >= 1
).astype(float)
df.loc[df["NewCategory"] != "Beverages", "less_healthy_npm"] = (
    pd.to_numeric(df.loc[df["NewCategory"] != "Beverages", "np_score"], errors="coerce") >= 4
).astype(float)

# Your list of indicators
Health_indicators = [
    "np_score", "less_healthy_npm",
    "total_a_points", "energy_points", "sat_fat_points",
    "total_sugar_points", "sodium_points",
    "total_c_points", "adjusted_c_points",
    "fvn_points", "fiber_points", "protein_points"
]

# Ensure numeric
df[Health_indicators] = df[Health_indicators].apply(pd.to_numeric, errors='coerce')

# ------------------------------------------------------------
# 2. Group and compute
# ------------------------------------------------------------
grp       = df.groupby("NewCategory")
count_ser = grp.size()
mean_df   = grp[Health_indicators].mean()
std_df    = grp[Health_indicators].std()

# Format mean (std)
formatted = pd.DataFrame(index=mean_df.index)
for col in Health_indicators:
    if col == "less_healthy_npm":
        # convert from share to percent
        formatted[col] = mean_df[col].combine(
            std_df[col],
            lambda m, s: f"{m * 100:.2f} ({s * 100:.2f})" if pd.notna(m) else ""
        )
    else:
        formatted[col] = mean_df[col].combine(
            std_df[col],
            lambda m, s: f"{m:.2f} ({s:.2f})" if pd.notna(m) else ""
        )

# Build summary_df
summary_df = formatted.copy()
summary_df.insert(0, "Number of New Products", count_ser)
summary_df = summary_df.reset_index().rename(columns={"NewCategory": "Category"})

# Add overall row
overall = {
    "Category": "All products",
    "Number of New Products": len(df)
}
for col in Health_indicators:
    m, s = df[col].mean(), df[col].std()
    if col == "less_healthy_npm":
        overall[col] = f"{m * 100:.2f} ({s * 100:.2f})"
    else:
        overall[col] = f"{m:.2f} ({s:.2f})"

summary_df = pd.concat(
    [summary_df, pd.DataFrame([overall])],
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Export into a .docx in landscape with table grid and merged headers
# ------------------------------------------------------------
doc = Document()
sec = doc.sections[-1]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = sec.page_height, sec.page_width

for attr in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, attr, Inches(0.5))

# Map raw column names → second-row subheaders
subhdr = {
    "Category": "Category",
    "Number of New Products": "Number of New Products",
    "np_score": "NP Score",
    "less_healthy_npm": "Less Healthy (%)",
    "total_a_points": "Total A Points",
    "energy_points": "Energy",
    "sat_fat_points": "Saturated Fat",
    "total_sugar_points": "Total Sugar",
    "sodium_points": "Sodium",
    "total_c_points": "Total C Points",
    "adjusted_c_points": "Adjusted Total C Points",
    "fvn_points": "Fruit, Veg & Nuts",
    "fiber_points": "Fiber",
    "protein_points": "Protein"
}

n_rows = summary_df.shape[0] + 2   # 2 header rows + data
n_cols = summary_df.shape[1]
table = doc.add_table(rows=n_rows, cols=n_cols)
table.style = "Table Grid"

# Top-level merges: (sr, sc, er, ec, text)
merges = [
    (0, 0, 1, 0, "Category"),
    (0, 1, 1, 1, "Number of New Products"),
    (0, 2, 1, 2, "NP Score"),
    (0, 3, 1, 3, "Less Healthy Under NPM"),
    (0, 4, 0, 8, "A Points"),
    (0, 9, 0, 13, "C Points"),
]

for sr, sc, er, ec, txt in merges:
    mc = table.cell(sr, sc).merge(table.cell(er, ec))
    mc.text = txt
    mc.paragraphs[0].alignment = 1

# Second-row subheaders
for j, col in enumerate(summary_df.columns):
    cell = table.cell(1, j)
    cell.text = subhdr[col]
    cell.paragraphs[0].alignment = 1

# Fill the data
for i, row in enumerate(summary_df.itertuples(index=False), start=2):
    for j, val in enumerate(row):
        table.cell(i, j).text = str(val)

# ------------------------------------------------------------
# 4. Save out
# ------------------------------------------------------------
os.chdir(RESULT_PATH)

out_docx = "Table1_less_healthy_summary.docx"
out_xlsx = "Table1_less_healthy_summary_by_Category.xlsx"

doc.save(out_docx)
summary_df.to_excel(out_xlsx, index=False)

print("✅ Written:", out_docx)
print("✅ Exported:", out_xlsx)

## 3.2 Figures

### Figure 1 Trends in Claims and Product Nutritional Quality (2015–2024)

In [ ]:
from pathlib import Path

MAIN_OUTPUT_ROOT = Path(RESULT_PATH) / "Final_Main_Figures"
MAIN_FIGURE_DIR = MAIN_OUTPUT_ROOT / "Figures"
SOURCE_DATA_DIR = MAIN_OUTPUT_ROOT / "Source_Data"
PANEL_DIR = MAIN_OUTPUT_ROOT / "Panels"
for _folder in (MAIN_FIGURE_DIR, SOURCE_DATA_DIR, PANEL_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

import matplotlib as mpl

# ========== Nature-style Font Settings ==========
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.size'] = 7
mpl.rcParams['axes.labelsize'] = 7
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 6

# ================== DATA PREP ==================
df = df.copy()

claim_count_columns = ['Num_Minus', 'Num_Plus', 'Num_Natural', 'Num_Functional']
claim_type_labels   = ['Minus Claims', 'Plus Claims', 'Natural Claims', 'Functional Claims']

df['Year'] = df['Year'].astype(int)
df['NewCategory'] = df['NewCategory'].astype(str)
df['np_score'] = pd.to_numeric(df['np_score'], errors='coerce')

# Construct Less Healthy under NPM
df['less_healthy_npm'] = np.where(
    df['NewCategory'] == 'Beverages',
    (df['np_score'] >= 1).astype(int),
    (df['np_score'] >= 4).astype(int)
)

line_data = []

# Claim lines
for col, label in zip(claim_count_columns, claim_type_labels):
    df[f'{col}_has_claim'] = df[col] > 0
    grouped = df.groupby('Year')[f'{col}_has_claim']
    mean = grouped.mean() * 100
    std = grouped.std() * 100
    count = grouped.count()
    ci95 = 1.96 * std / np.sqrt(count)

    for year in mean.index:
        line_data.append({
            'Year': year,
            'Display Label': label,
            'Share (%)': float(f"{mean[year]:.1f}"),
            'Std Dev': float(f"{std[year]:.1f}"),
            'CI Lower': float(f"{mean[year] - ci95[year]:.1f}"),
            'CI Upper': float(f"{mean[year] + ci95[year]:.1f}"),
            'Label': f"{mean[year]:.1f}"
        })

# Any claim line
df['AnyClaim'] = df[claim_count_columns].sum(axis=1) > 0
grouped = df.groupby('Year')['AnyClaim']
mean = grouped.mean() * 100
std = grouped.std() * 100
count = grouped.count()
ci95 = 1.96 * std / np.sqrt(count)

for year in mean.index:
    line_data.append({
        'Year': year,
        'Display Label': 'Any Studied On-Package Claim',
        'Share (%)': float(f"{mean[year]:.1f}"),
        'Std Dev': float(f"{std[year]:.1f}"),
        'CI Lower': float(f"{mean[year] - ci95[year]:.1f}"),
        'CI Upper': float(f"{mean[year] + ci95[year]:.1f}"),
        'Label': f"{mean[year]:.1f}"
    })

# Less healthy bars
grouped = df.groupby('Year')['less_healthy_npm']
mean = grouped.mean() * 100
std = grouped.std() * 100
count = grouped.count()
ci95 = 1.96 * std / np.sqrt(count)

for year in mean.index:
    line_data.append({
        'Year': year,
        'Display Label': 'Less Healthy Under NPM',
        'Share (%)': float(f"{mean[year]:.1f}"),
        'Std Dev': float(f"{std[year]:.1f}"),
        'CI Lower': float(f"{mean[year] - ci95[year]:.1f}"),
        'CI Upper': float(f"{mean[year] + ci95[year]:.1f}"),
        'Label': f"{mean[year]:.1f}"
    })

plot_df_all_augmented = pd.DataFrame(line_data)


# ================== FINAL FIGURE 1 ==================
years = np.array(sorted(plot_df_all_augmented['Year'].unique()))
display_map = {
    'Less Healthy Under NPM': 'Less healthy under the NPM',
    'Any Studied On-Package Claim': 'Any studied on-package claim',
    'Minus Claims': 'Minus claims',
    'Natural Claims': 'Natural claims',
    'Plus Claims': 'Plus claims',
    'Functional Claims': 'Functional claims',
}
series = {}
for _old_label, _new_label in display_map.items():
    _values = (plot_df_all_augmented
               .loc[plot_df_all_augmented['Display Label'] == _old_label]
               .set_index('Year')['Share (%)']
               .reindex(years))
    series[_new_label] = _values.to_numpy(dtype=float)

_source_rows = []
for _label, _values in series.items():
    _source_rows.extend({"Year": int(_year), "Series": _label,
                         "Share of products (%)": float(_value)}
                        for _year, _value in zip(years, _values))
figure1_source = pd.DataFrame(_source_rows)
figure1_source.to_csv(
    SOURCE_DATA_DIR / "Figure1_SourceData_DisplayedValues.csv", index=False
)

fig, ax = plt.subplots(figsize=(10.0, 5.9))
x = np.arange(len(years))
ax.bar(x, series['Less healthy under the NPM'], width=0.46,
       color='#BDBDBD', edgecolor='white', linewidth=0.7,
       label='Less healthy under the NPM', zorder=1)
styles = {
    'Any studied on-package claim': ('#55A868', 'o', '--'),
    'Minus claims': ('#DD8452', 'x', '-'),
    'Natural claims': ('#4C72B0', 'D', '-'),
    'Plus claims': ('#8172B3', '^', '-'),
    'Functional claims': ('#CC78BC', 's', '-'),
}
for _label, (_color, _marker, _linestyle) in styles.items():
    _values = series[_label]
    ax.plot(x, _values, color=_color, marker=_marker, linestyle=_linestyle,
            linewidth=2.2, markersize=6.5, markeredgewidth=1.1,
            label=_label, zorder=3)
    for _x, _value in zip(x, _values):
        ax.text(_x, _value + 1.15, f"{_value:.1f}", color=_color,
                ha='center', va='bottom', fontsize=7.0)
for _x, _value in zip(x, series['Less healthy under the NPM']):
    ax.text(_x, _value + 1.0, f"{_value:.1f}", color='#4A4A4A',
            ha='center', va='bottom', fontsize=7.0)
ax.set_xlim(-0.7, len(years) - 0.3)
ax.set_ylim(0, 75)
ax.set_xticks(x, years)
ax.set_xlabel('Year', fontsize=9.0)
ax.set_ylabel('Share of products (%)', fontsize=9.0)
ax.tick_params(axis='both', labelsize=7.5, length=3)
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
legend_order = [
    'Less healthy under the NPM', 'Natural claims',
    'Any studied on-package claim', 'Plus claims',
    'Minus claims', 'Functional claims',
]
ax.legend([by_label[_label] for _label in legend_order], legend_order,
          loc='upper center', bbox_to_anchor=(0.5, -0.105), ncol=3,
          frameon=False, fontsize=7.5, columnspacing=1.5, handlelength=2.4)
fig.subplots_adjust(left=0.08, right=0.985, top=0.965, bottom=0.205)
for _ext in ('pdf', 'svg', 'png'):
    _kwargs = {'dpi': 300} if _ext == 'png' else {}
    fig.savefig(MAIN_FIGURE_DIR / f'Figure1_Trend_of_Claims_and_Less_Healthy.{_ext}', **_kwargs)
plt.show()


### Figure 2. Heatmap of Claims Popularity by Product Category and Specific Claims

In [ ]:
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from scipy.stats import binom

# STEP 2: Define claim groups
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

# STEP 3: Human-readable claim labels
claim_vars = [
    'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
    'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
    'LessChol', 'LessGlycemic', 'PlusVitamin', 'HighProtein', 'AddedCalcium',
    'HighFiber',  'AllNatural', 'NoArtAdditives',
    'NoArtColourings', 'NoArtFlavourings', 'NoArtPreservatives',
    'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain',
    'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
    'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
]

claim_labels = [
    'Low/No/Reduced Calorie', 'No Added Sugar', 'Sugar Free', 'Low/Reduced Sugar', 'Diet/Light',
    'Low/No/Reduced Sodium', 'Low/No/Reduced Carb', 'Low/No/Reduced Fat', 'Low/No/Reduced Trans Fat', 'Low/No/Reduced Saturated Fat',
    'Low/No/Reduced Cholesterol', 'Low/No/Reduced Glycemic', 'Vitamin/Mineral Fortified', 'High/Added Protein', 'Added Calcium',
    'High/Added Fiber', 'All Natural Product', 'No Added/Artificial Additives',
    'No Added/Artificial Colourings', 'No Added/Artificial Flavourings', 'No Added/Artificial Preservatives',
    'No Additives/Preservatives', 'GMO Free', 'Organic', 'Whole Grain', 
    'Brain & Nervous System','Immune System', 'Digestive', 'Probiotic/Prebiotic', 
    'Antioxidant', 'Weight & Muscle Gain','Cardiovascular', 'Bone/Skin/Nails&Hair/Eye Health'
]

# STEP 4: Create label map for display
full_label_map = {}
idx = 0
for group, claims in claim_groups.items():
    for claim in claims:
        full_label_map[claim] = f"{group}: {claim_labels[idx]}"
        idx += 1

# STEP 5: Compute share and CI
df_claims = df[['NewCategory'] + claim_vars].copy()
df_claims['NewCategory'] = df_claims['NewCategory'].astype(str)

# Count total products per category
category_counts = df_claims.groupby('NewCategory').size()

# Compute mean and CI
means = df_claims.groupby('NewCategory')[claim_vars].mean()
counts = df_claims.groupby('NewCategory')[claim_vars].sum()

ci_upper = {}
ci_lower = {}
ci_text = pd.DataFrame(index=means.index, columns=means.columns)

for cat in means.index:
    n = category_counts[cat]
    for claim in claim_vars:
        p = means.loc[cat, claim]
        x = counts.loc[cat, claim]
        ci = 1.96 * np.sqrt(p * (1 - p) / n)
        percent = p * 100
        ci_percent = ci * 100
        ci_text.loc[cat, claim] = f"{percent:.1f}±{ci_percent:.1f}"

# STEP 6: Transpose for heatmap
df_transposed = ci_text.T
df_transposed.rename(index=full_label_map, inplace=True)
cleaned_claim_names_y = [label.split(": ", 1)[1] for label in df_transposed.index]
group_names_y = [label.split(":")[0] for label in df_transposed.index]

# STEP 7: Identify group boundaries
group_boundaries_y = []
prev_group = None
for i, label in enumerate(df_transposed.index):
    group = label.split(":")[0]
    if group != prev_group:
        group_boundaries_y.append(i)
        prev_group = group
group_boundaries_y.append(len(df_transposed.index))


# STEP 8: Final Figure 2 heatmap
heatmap_values = means.T
heatmap_values.rename(index=full_label_map, inplace=True)
heatmap_values *= 100
claims = cleaned_claim_names_y
categories = list(df_transposed.columns)

_figure2_rows = []
for _claim in df_transposed.index:
    for _category in df_transposed.columns:
        _value_text = str(df_transposed.loc[_claim, _category])
        _share, _margin = [float(_part) for _part in _value_text.split("±")]
        _figure2_rows.append({
            "Claim": _claim.split(": ", 1)[-1],
            "Product Category": _category,
            "Estimated Share (%)": _share,
            "95% CI Margin (percentage points)": _margin,
            "95% CI Lower (%)": _share - _margin,
            "95% CI Upper (%)": _share + _margin,
        })
figure2_source = pd.DataFrame(_figure2_rows)
figure2_source.to_csv(
    SOURCE_DATA_DIR / "Figure2_SourceData_DisplayedValues.csv", index=False
)

fig = plt.figure(figsize=(8.0, 11.0))
gs = fig.add_gridspec(1, 2, width_ratios=[24, 1], wspace=0.05,
                      left=0.275, right=0.94, top=0.985, bottom=0.15)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])
sns.heatmap(heatmap_values.to_numpy(dtype=float),
            annot=df_transposed.to_numpy(), fmt='', cmap='YlGnBu',
            vmin=0, vmax=60, linewidths=0.35, linecolor='#F2F2F2',
            annot_kws={'fontsize': 6.15}, cbar_ax=cax,
            cbar_kws={'ticks': np.arange(0, 61, 10)}, ax=ax)
colorbar = ax.collections[0].colorbar
if colorbar.solids is not None:
    colorbar.solids.set_rasterized(False)
    colorbar.solids.set_edgecolor('face')
    colorbar.solids.set_linewidth(0)
colorbar.ax.tick_params(labelsize=7.0, length=2.5)
colorbar.ax.set_ylabel('Share of products with claim (%)', fontsize=7.3)
ax.set_xticks(np.arange(len(categories)) + 0.5)
ax.set_xticklabels(categories, rotation=90, ha='center', fontsize=7.0)
ax.set_yticks(np.arange(len(claims)) + 0.5)
ax.set_yticklabels(claims, rotation=0, fontsize=7.0)
ax.tick_params(axis='both', length=0)
ax.set_xlabel('Product categories', fontsize=8.0, fontweight='bold', labelpad=7)
ax.set_ylabel('')
boundaries = [0, 12, 16, 25, 33]
groups = ['Minus', 'Plus', 'Natural', 'Functional']
for _start, _end, _group in zip(boundaries[:-1], boundaries[1:], groups):
    ax.add_patch(plt.Rectangle((0, _start), len(categories), _end - _start,
                              fill=False, edgecolor='black', linewidth=0.8,
                              clip_on=False))
    _midpoint = (_start + _end) / 2
    _y_fig = 0.985 - (_midpoint / len(claims)) * (0.985 - 0.15)
    fig.text(0.060, _y_fig, _group, rotation=90,
             ha='center', va='center', fontsize=7.5, fontweight='bold')
fig.text(0.018, (0.985 + 0.15) / 2, 'Claims', rotation=90,
         ha='center', va='center', fontsize=8.0, fontweight='bold')
for _ext in ('pdf', 'svg', 'png'):
    _kwargs = {'dpi': 300} if _ext == 'png' else {}
    fig.savefig(MAIN_FIGURE_DIR / f'Figure2_Heatmap_ClaimsPopularity_WithCI.{_ext}', **_kwargs)
plt.show()


### Figure 3 Nutrition Profile Score with and without Claims

#### Figure3A. Discrepancy of Claims for Food Products

In [ ]:
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 0. Use Food-only data
# ------------------------------------------------------------
df_nobeve = df.loc[df['NewCategory'] != 'Beverages'].copy()

# ------------------------------------------------------------
# 1. Define claim groups and readable labels
# ------------------------------------------------------------
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree',
        'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = {
    'LessCalorie': 'Low/No/Reduced Calorie',
    'NoAddedSugar': 'No Added Sugar',
    'SugarFree': 'Sugar Free',
    'LowSugar': 'Low/Reduced Sugar',
    'Diet': 'Diet/Light',
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessCarb': 'Low/No/Reduced Carb',
    'LessFat': 'Low/No/Reduced Fat',
    'LessTransFat': 'Low/No/Reduced Trans Fat',
    'LessSatFat': 'Low/No/Reduced Saturated Fat',
    'LessChol': 'Low/No/Reduced Cholesterol',
    'LessGlycemic': 'Low/No/Reduced Glycemic',

    'PlusVitamin': 'Vitamin/Mineral Fortified',
    'HighProtein': 'High/Added Protein',
    'AddedCalcium': 'Added Calcium',
    'HighFiber': 'High/Added Fiber',

    'AllNatural': 'All Natural Product',
    'NoArtAdditives': 'No Added/Artificial Additives',
    'NoArtColourings': 'No Added/Artificial Colourings',
    'NoArtFlavourings': 'No Added/Artificial Flavourings',
    'NoArtPreservatives': 'No Added/Artificial Preservatives',
    'NoAdditivesPreservatives': 'No Additives/Preservatives',
    'GMOFree': 'GMO Free',
    'Organic': 'Organic',
    'Wholegrain': 'Whole Grain',

    'BrainNervSystem': 'Brain & Nervous System',
    'ImmuneSystem': 'Immune System',
    'Digestive': 'Digestive',
    'Probiotic': 'Probiotic/Prebiotic',
    'Antioxidant': 'Antioxidant',
    'WgtMuscle': 'Weight & Muscle Gain',
    'Cardiovascular': 'Cardiovascular',
    'BoneSkinHairEyeHealth': 'Bone/Skin/Nails & Hair/Eye Health'
}

claims = [c for grp in claim_groups.values() for c in grp]

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def get_significance_marker(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

def assign_group(claim):
    for group, items in claim_groups.items():
        if claim in items:
            return group
    return 'Other'

# ------------------------------------------------------------
# 3. Compute Food-only results
# ------------------------------------------------------------
results = []

for claim in claims:
    if claim not in df_nobeve.columns:
        continue

    with_claim = df_nobeve.loc[df_nobeve[claim] == 1, 'np_score'].dropna()
    without_claim = df_nobeve.loc[df_nobeve[claim] == 0, 'np_score'].dropna()

    popularity = df_nobeve[claim].mean() * 100 if claim in df_nobeve.columns else np.nan

    mean_with = with_claim.mean() if len(with_claim) > 0 else np.nan
    mean_without = without_claim.mean() if len(without_claim) > 0 else np.nan
    direction = mean_with - mean_without if pd.notna(mean_with) and pd.notna(mean_without) else np.nan

    mw_stat, p_val = np.nan, np.nan
    if len(with_claim) > 0 and len(without_claim) > 0:
        try:
            mw_stat, p_val = mannwhitneyu(
                with_claim,
                without_claim,
                alternative='two-sided'
            )
        except ValueError:
            pass

    results.append({
        'Claim': claim,
        'Claim Label': claim_labels[claim],
        'With Claim': mean_with,
        'Without Claim': mean_without,
        'Mean Difference': direction,
        'Popularity': popularity,
        'MW Statistic': mw_stat,
        'p_value': p_val,
        'N With Claim': len(with_claim),
        'N Without Claim': len(without_claim)
    })

results_df_nobeve = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Add significance, colors, flags, groups
# ------------------------------------------------------------
results_df_nobeve['Group'] = results_df_nobeve['Claim'].apply(assign_group)
results_df_nobeve['Significance'] = results_df_nobeve['p_value'].apply(get_significance_marker)
results_df_nobeve['Direction'] = results_df_nobeve['With Claim'] - results_df_nobeve['Without Claim']

deep_palette = sns.color_palette('deep')
blue_deep = deep_palette[0]
orange_deep = deep_palette[1]

results_df_nobeve['Color'] = results_df_nobeve.apply(
    lambda row: orange_deep if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else
                blue_deep if pd.notna(row['Direction']) and row['Direction'] < 0 and row['p_value'] < 0.05 else
                'black',
    axis=1
)

results_df_nobeve['Label'] = results_df_nobeve.apply(
    lambda row: f"{row['Claim Label']} {row['Significance']}"
    if pd.notna(row['p_value']) and row['p_value'] < 0.05
    else row['Claim Label'],
    axis=1
)

results_df_nobeve['Flag'] = results_df_nobeve.apply(
    lambda row: True if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else False,
    axis=1
)

results_df = results_df_nobeve.copy()

# ------------------------------------------------------------
# 5. Save Food table
# ------------------------------------------------------------
table_to_save = results_df_nobeve[[
    'Group', 'Claim', 'Claim Label', 'With Claim', 'Without Claim',
    'Mean Difference', 'Popularity', 'MW Statistic', 'p_value',
    'Significance', 'N With Claim', 'N Without Claim', 'Flag'
]].copy()

csv_file = os.path.join(SOURCE_DATA_DIR, "Figure3A_Discrepancy_Avg_NP_Score_Food_table.csv")
excel_file = os.path.join(SOURCE_DATA_DIR, "Figure3A_Discrepancy_Avg_NP_Score_Food_table.xlsx")

table_to_save.to_csv(csv_file, index=False)
table_to_save.to_excel(excel_file, index=False)

# ------------------------------------------------------------
# 6. Melt for plotting
# ------------------------------------------------------------
results_melted = results_df_nobeve.melt(
    id_vars=['Claim', 'Popularity', 'p_value', 'Label', 'Color', 'Flag', 'Group'],
    value_vars=['With Claim', 'Without Claim'],
    var_name='Claim Status',
    value_name='Average NPM Score'
)

label_order = results_df_nobeve['Label'].tolist()
results_melted['Label'] = pd.Categorical(
    results_melted['Label'],
    categories=label_order,
    ordered=True
)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------
sns.set_theme(style="white")
fig = plt.figure(figsize=(9, 9))

scatter = sns.scatterplot(
    data=results_melted,
    x='Average NPM Score',
    y='Label',
    hue='Claim Status',
    size='Popularity',
    sizes=(50, 400),
    palette='deep',
    style='Claim Status',
    legend='brief'
)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(0.75)

# color y labels by direction/significance
label_color_map = dict(zip(results_df_nobeve['Label'], results_df_nobeve['Color']))
for tick in ax.get_yticklabels():
    txt = tick.get_text()
    if txt in label_color_map:
        tick.set_color(label_color_map[txt])

# ------------------------------------------------------------
# 8. Claim group spans and group labels
# ------------------------------------------------------------
label_to_group = dict(zip(results_df_nobeve['Label'], results_df_nobeve['Group']))
spans = []
prev_g, start_i = None, None

for i, lab in enumerate(label_order):
    g = label_to_group.get(lab, 'Other')
    if g != prev_g:
        if prev_g is not None:
            spans.append((start_i, i, prev_g))
        start_i = i
        prev_g = g
if prev_g is not None:
    spans.append((start_i, len(label_order), prev_g))

x0, x1 = ax.get_xlim()
for j, (s, e, g) in enumerate(spans):
    y0 = s - 0.5
    height = e - s
    if j != 0 and j != len(spans) - 1:
        ax.add_patch(
            patches.Rectangle(
                (x0, y0),
                width=(x1 - x0),
                height=height,
                fill=False,
                edgecolor='black',
                linewidth=1.2,
                zorder=1
            )
        )

xspan = x1 - x0
for (s, e, g) in spans:
    ax.text(
        x0 - 0.5 * xspan,
        (s + e - 1) / 2,
        g,
        ha='right',
        va='center',
        fontsize=12,
        rotation=90,
        fontweight='bold',
        color='black'
    )

# ------------------------------------------------------------
# 9. Vertical cutoff line and label
# ------------------------------------------------------------
cutoff_color = '#c44e52'
ax.axvline(
    x=4,
    color=cutoff_color,
    linestyle='--',
    linewidth=2,
    zorder=2
)

x0, x1 = ax.get_xlim()
ymin, ymax = ax.get_ylim()
yrange = ymax - ymin
x_cut = 4

ax.text(
    x_cut + 0.6,              
    ymin - 0.045 * yrange,    
    "NPM threshold     \n(≥4 for “less healthy”)",
    color=cutoff_color,
    fontsize=9,
    ha='right',         
    va='center',       
    zorder=3
)
# ------------------------------------------------------------
# 10. Formatting
# ------------------------------------------------------------
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.1f}'))

plt.xlabel("Average nutrient profile score", fontsize=12)
plt.ylabel("Claims", fontweight='bold', fontsize=12, labelpad=30)

ax.set_axisbelow(True)
ax.grid(
    True, which='both', axis='both',
    linestyle='--', linewidth=0.8, alpha=0.65, color='gray'
)

plt.tight_layout()

handles, labels = scatter.get_legend_handles_labels()
filtered = [(h, l) for h, l in zip(handles, labels) if l in ['With Claim', 'Without Claim']]
if filtered:
    h, l = zip(*filtered)
    l = ['With claim' if x == 'With Claim' else 'Without claim' for x in l]
    plt.legend(
        h, l,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.115),
        ncol=2,
        fontsize=12,
        frameon=True
    )

out = os.path.join(PANEL_DIR, "Figure3A_Discrepancy_Avg_NP_Score_Food.png")
out_pdf = out.replace(".png", ".pdf")

for spine in ax.spines.values():
    spine.set_linewidth(0.75)
plt.savefig(out, dpi=1200, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(os.path.join(PANEL_DIR, "Figure3A.svg"), bbox_inches="tight")
plt.show()

print(f"Saved figure: {out}")
print(f"Saved CSV: {csv_file}")
print(f"Saved Excel: {excel_file}")
print(table_to_save.head())


#### Figure3B. Components Contribution to Discrepancy

In [ ]:
# ============================================================
# Figure 3B (Food): minimally revised bar + point + 95% CI
# ============================================================
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu

food_folder = str(PANEL_DIR / "Food")
os.makedirs(food_folder, exist_ok=True)
df_food = df.loc[df['NewCategory'] != 'Beverages'].copy()

blue_deep = '#4C72B0'
orange_deep = '#DD8452'
neutral_gray = '#666666'

a_components = ['energy_points', 'sat_fat_points', 'total_sugar_points', 'sodium_points']
c_components = ['fvn_points', 'fiber_points', 'protein_points']
all_components = a_components + c_components

component_label_map = {
    'energy_points': 'Energy',
    'sat_fat_points': 'Saturated Fat',
    'total_sugar_points': 'Total Sugar',
    'sodium_points': 'Sodium',
    'fvn_points': 'Fruit, Veg & Nuts',
    'fiber_points': 'Fiber',
    'protein_points': 'Protein'
}

def significance_marker(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return ''

def component_results(df_subset, claim, panel_name):
    rows = []
    for comp in all_components:
        x = df_subset.loc[df_subset[claim] == 1, comp].dropna()
        y = df_subset.loc[df_subset[claim] == 0, comp].dropna()
        mean_with = x.mean() if len(x) else np.nan
        mean_without = y.mean() if len(y) else np.nan
        mean_diff = mean_with - mean_without
        u_stat, p_val = (np.nan, np.nan)
        if len(x) and len(y):
            u_stat, p_val = mannwhitneyu(x, y, alternative='two-sided')
        if len(x) > 1 and len(y) > 1:
            se = np.sqrt(x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))
            margin = 1.96 * se
        else:
            margin = np.nan
        contribution = mean_diff if comp in a_components else -mean_diff
        rows.append({
            'Panel': panel_name,
            'Claim': claim,
            'Component': comp,
            'Component Label': component_label_map[comp],
            'Mean With Claim': mean_with,
            'Mean Without Claim': mean_without,
            'Mean Difference': mean_diff,
            'Signed NPM Contribution': contribution,
            'CI Lower': mean_diff - margin,
            'CI Upper': mean_diff + margin,
            'CI Margin': margin,
            'N With Claim': len(x),
            'N Without Claim': len(y),
            'U Statistic': u_stat,
            'Exact P Value': p_val,
            'Test': 'Two-sided Mann-Whitney U (Wilcoxon rank-sum)',
            'Multiple Testing Adjustment': 'None',
            'Significance': significance_marker(p_val)
        })
    out = pd.DataFrame(rows)
    out['Absolute Contribution'] = out['Signed NPM Contribution'].abs()
    return out.sort_values('Absolute Contribution', ascending=False).reset_index(drop=True)

food_specs = [
    ('Dairy & Eggs', df_food['NewCategory'] == 'Dairy & Eggs'),
    ('Meals & Side Dishes', df_food['NewCategory'] == 'Meals & Side Dishes')
]
food_results = [component_results(df_food.loc[mask], 'LessTransFat', category)
                for category, mask in food_specs]
food_source = pd.concat(food_results, ignore_index=True)
food_source.to_csv(os.path.join(food_folder, 'Figure3B_SourceData.csv'), index=False)
food_source.to_excel(os.path.join(food_folder, 'Figure3B_SourceData.xlsx'), index=False)

x_bound = 2.6

sns.set_theme(style='white')
fig, axes = plt.subplots(1, 2, figsize=(11.6, 6.2), sharex=True)
for ax, (category, _), comp_df in zip(axes, food_specs, food_results):
    y = np.arange(len(comp_df))
    colors = [orange_deep if v > 0 else blue_deep if v < 0 else neutral_gray
              for v in comp_df['Signed NPM Contribution']]
    # Estimate plot: dot = mean difference; whisker = normal-approximation 95% CI.
    for i, row in comp_df.iterrows():
        c = colors[i]
        ax.errorbar(row['Mean Difference'], i, xerr=row['CI Margin'], fmt='o',
                    color=c, ecolor=c, markersize=5.5, elinewidth=1.4,
                    capsize=3.5, capthick=1.2, zorder=4)
        margin = row['CI Margin'] if pd.notna(row['CI Margin']) else np.nan
        estimate = 0.0 if abs(row['Mean Difference']) < 0.005 else row['Mean Difference']
        value = f"{estimate:.2f} ± {margin:.2f}"
        ax.text(1.04, i, value, transform=ax.get_yaxis_transform(),
                ha='left', va='center', fontsize=7.2, color='#333333', clip_on=False)
    labels = [f"{r['Component Label']} {r['Significance']}".strip()
              for _, r in comp_df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=1.0, zorder=3)
    ax.set_xlim(-x_bound, x_bound)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1.0))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.grid(True, axis='both', linestyle='--', color='#AAAAAA', linewidth=0.45, alpha=0.75)
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
    ax.set_title(f"Low/No/Reduced Trans Fat\n({category})", fontsize=11)
    ax.set_xlabel('Mean difference in component score\n(with claim − without claim)', fontsize=9)
    ax.text(1.04, 1.015, 'Mean difference\n± 95% CI margin', transform=ax.transAxes,
            ha='left', va='bottom', fontsize=7.0, fontweight='bold')
axes[0].set_ylabel('Components of nutrient profile score', fontsize=9, fontweight='bold')
fig.subplots_adjust(left=0.15, right=0.86, bottom=0.14, top=0.86, wspace=0.95)

for ext in ('png', 'pdf', 'svg'):
    path = os.path.join(food_folder, f'Figure3B_Food_Component_Discrepancy_Review.{ext}')
    fig.savefig(path, dpi=600 if ext == 'png' else None, bbox_inches='tight')
plt.show()
print(food_source[['Panel','Component Label','Mean Difference','CI Lower','CI Upper',
                   'N With Claim','N Without Claim','U Statistic','Exact P Value']])


#### Figure 3. Combine Figure3A & Figure 3B

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BLUE = "#4C72B0"
ORANGE = "#DD8452"
GRAY = "#666666"
GRID = "#D9D9D9"
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "axes.linewidth": 0.75,
    "axes.edgecolor": "black",
})

def clean_zero(value):
    return 0.0 if abs(value) < 0.005 else value

def plot_panel_a(ax, data, threshold, category_label):
    data = data.reset_index(drop=True).copy()
    data["Significance"] = data["Significance"].fillna("")
    y = np.arange(len(data))
    pop = data["Popularity"].astype(float).to_numpy()
    sizes = 11 + 60 * (pop - pop.min()) / max(pop.max() - pop.min(), 1e-9)

    ax.scatter(data["With Claim"], y, s=sizes, color=BLUE, marker="o",
               linewidth=0, label="With claim", zorder=3)
    ax.scatter(data["Without Claim"], y, s=sizes, color=ORANGE, marker="x",
               linewidth=1.2, label="Without claim", zorder=3)

    labels = []
    label_colors = []
    for _, row in data.iterrows():
        labels.append(f"{row['Claim Label']} {row['Significance']}".strip())
        if row["p_value"] < 0.05 and row["Mean Difference"] > 0:
            label_colors.append(ORANGE)
        elif row["p_value"] < 0.05 and row["Mean Difference"] < 0:
            label_colors.append(BLUE)
        else:
            label_colors.append("black")

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=6.5)
    for tick, color in zip(ax.get_yticklabels(), label_colors):
        tick.set_color(color)
    ax.invert_yaxis()

    groups = data["Group"].astype(str).tolist()
    starts = [0]
    for i in range(1, len(groups)):
        if groups[i] != groups[i - 1]:
            starts.append(i)
            ax.axhline(i - 0.5, color="black", linewidth=0.8)
    starts.append(len(groups))
    for start, end in zip(starts[:-1], starts[1:]):
        ax.text(-0.43, (start + end - 1) / 2, groups[start],
                transform=ax.get_yaxis_transform(), rotation=90,
                ha="center", va="center", fontsize=6.5, fontweight="bold")

    all_x = np.r_[data["With Claim"].to_numpy(), data["Without Claim"].to_numpy(), threshold]
    span = all_x.max() - all_x.min()
    ax.set_xlim(all_x.min() - 0.08 * span, all_x.max() + 0.08 * span)
    ax.axvline(threshold, color="#C44E52", linestyle="--", linewidth=1.1)
    ax.grid(True, axis="both", linestyle="--", linewidth=0.45, color="#AAAAAA", alpha=0.75)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
    ax.set_xlabel("Average nutrient profile score", fontsize=7.2)
    ax.set_ylabel("")
    ax.text(-0.47, 0.5, "Claims", transform=ax.transAxes, rotation=90,
            ha="center", va="center", fontsize=7.2, fontweight="bold",
            clip_on=False)
    ax.tick_params(axis="x", labelsize=6.2, length=2.5)
    ax.tick_params(axis="y", length=0)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=2,
              frameon=True, fontsize=6.3, handletextpad=0.4, columnspacing=1.2)
    comparator = "≥4" if threshold == 4 else "≥1"
    ax.text(0.995, 1.012, f'NPM "less healthy" threshold (score {comparator})',
            transform=ax.transAxes, color="#C44E52",
            ha="right", va="bottom", fontsize=5.9)

def contribution_color(row):
    if float(row["Exact P Value"]) >= 0.05:
        return GRAY
    contribution = float(row["Signed NPM Contribution"])
    if contribution > 0:
        return ORANGE
    if contribution < 0:
        return BLUE
    return GRAY

def plot_panel_b(ax, data, xlim, show_ylabel=False, show_xlabel=True):
    data = data.reset_index(drop=True).copy()
    data["Significance"] = data["Significance"].fillna("")
    y = np.arange(len(data))
    for i, row in data.iterrows():
        color = contribution_color(row)
        ax.errorbar(float(row["Mean Difference"]), i,
                    xerr=float(row["CI Margin"]), fmt="o",
                    color=color, ecolor=color, markersize=3.6,
                    elinewidth=1.0, capsize=2.4, capthick=0.9, zorder=3)

    labels = [f"{row['Component Label']} {row['Significance']}".strip()
              for _, row in data.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=5.8)
    for tick, (_, row) in zip(ax.get_yticklabels(), data.iterrows()):
        tick.set_color(contribution_color(row) if float(row["Exact P Value"]) < 0.05 else "black")
    ax.invert_yaxis()
    ax.set_xlim(*xlim)
    ax.axvline(0, color="black", linewidth=0.9)
    # Match Panel A's light dashed grid while retaining the solid black zero line.
    ax.grid(True, axis="both", linestyle="--", linewidth=0.45,
            color="#AAAAAA", alpha=0.75)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
    ax.tick_params(axis="x", labelsize=5.7, length=2.5)
    ax.tick_params(axis="y", length=0)
    if show_xlabel:
        ax.set_xlabel("Mean difference in component score\n(with claim − without claim)", fontsize=6.1)
    if show_ylabel:
        ax.set_ylabel("Components of nutrient profile score", fontsize=6.0)

def plot_value_column(ax, data, show_header=True):
    data = data.reset_index(drop=True)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(data) - 0.5)
    ax.invert_yaxis()
    for i, row in data.iterrows():
        estimate = clean_zero(float(row["Mean Difference"]))
        lower = clean_zero(float(row["CI Lower"]))
        upper = clean_zero(float(row["CI Upper"]))
        ax.text(0, i, f"{estimate:.2f} ({lower:.2f} to {upper:.2f})",
                ha="left", va="center", fontsize=4.9, color="#333333")
    if show_header:
        ax.text(0, 1.018, "Mean difference (95% CI)",
                transform=ax.transAxes, ha="left", va="bottom",
                fontsize=5.0, fontweight="bold")
    ax.axis("off")

def panel_heading(fig, x, y, letter, title):
    fig.text(x, y, letter, fontsize=8.6, fontweight="bold", ha="left", va="top")
    fig.text(x + 0.023, y, title, fontsize=8.0, ha="left", va="top")

# Final Figure 3: live notebook results, landscape A4, vector output.
figure3a_plot = results_df_nobeve.copy()
figure3a_source = figure3a_plot[[
    'Group', 'Claim Label', 'With Claim', 'Without Claim',
    'Mean Difference', 'Popularity', 'MW Statistic', 'p_value',
    'Significance', 'N With Claim', 'N Without Claim'
]].copy().rename(columns={
    'Group': 'Claim Group',
    'Claim Label': 'Claim',
    'With Claim': 'Mean NP Score With Claim',
    'Without Claim': 'Mean NP Score Without Claim',
    'Mean Difference': 'Mean Difference (With - Without)',
    'Popularity': 'Claim Prevalence (%)',
    'MW Statistic': 'Mann-Whitney U Statistic',
    'p_value': 'Two-sided P Value',
    'N With Claim': 'N With Claim',
    'N Without Claim': 'N Without Claim',
})
figure3b_source = food_source[[
    'Panel', 'Component Label', 'Mean With Claim', 'Mean Without Claim',
    'Mean Difference', 'CI Lower', 'CI Upper', 'N With Claim',
    'N Without Claim', 'U Statistic', 'Exact P Value', 'Significance'
]].copy().rename(columns={
    'Panel': 'Category/Subcategory',
    'Component Label': 'NPM Component',
    'Mean With Claim': 'Mean Component Score With Claim',
    'Mean Without Claim': 'Mean Component Score Without Claim',
    'Mean Difference': 'Mean Difference (With - Without)',
    'CI Lower': '95% CI Lower',
    'CI Upper': '95% CI Upper',
    'U Statistic': 'Mann-Whitney U Statistic',
    'Exact P Value': 'Two-sided P Value',
})
figure3b_source.insert(0, 'Claim', 'Low/No/Reduced Trans Fat')
figure3a_source.to_csv(SOURCE_DATA_DIR / "Figure3A_SourceData.csv", index=False)
figure3b_source.to_csv(SOURCE_DATA_DIR / "Figure3B_SourceData.csv", index=False)

fig = plt.figure(figsize=(11.69, 8.27))
outer = fig.add_gridspec(1, 2, width_ratios=[1.42, 1.0],
                         left=0.245, right=0.94, bottom=0.12, top=0.89,
                         wspace=0.42)
ax_a = fig.add_subplot(outer[0])
plot_panel_a(ax_a, figure3a_plot, 4, "Food")
panel_heading(fig, 0.075, 0.955, "a",
              "Nutrient profile scores by claim status: market-level patterns")

sub = outer[1].subgridspec(2, 2, width_ratios=[3.1, 1.35],
                           wspace=0.08, hspace=0.30)
for i, panel in enumerate(["Dairy & Eggs", "Meals & Side Dishes"]):
    ax = fig.add_subplot(sub[i, 0])
    block = food_source.loc[food_source["Panel"] == panel]
    plot_panel_b(ax, block, (-2.6, 2.6), show_ylabel=False,
                 show_xlabel=(i == len(["Dairy & Eggs", "Meals & Side Dishes"]) - 1))
    ax.set_title(f"Low/No/Reduced Trans Fat\n({panel})", fontsize=6.4, pad=5)
    value_ax = fig.add_subplot(sub[i, 1])
    plot_value_column(value_ax, block, show_header=(i == 0))
panel_heading(fig, 0.628, 0.955, "b",
              "Nutrient contributions to discrepancy within categories")
fig.text(0.640, 0.505, "Components of nutrient profile score",
         rotation=90, ha="center", va="center", fontsize=6.0,
         fontweight="bold")

for ext in ("pdf", "svg", "png"):
    kwargs = {"dpi": 300} if ext == "png" else {}
    fig.savefig(MAIN_FIGURE_DIR / f"Figure3_Discrepancy_Food.{ext}", **kwargs)
plt.show()


## Figure 4 Beverages 

#### Figure4A. Discrepancy of Claims for Beverages

In [ ]:
# ============================================================
# Figure 4A: Beverages
# Average NP score for products with vs without claims
# Wilcoxon rank-sum test
# vertical cutoff line at NP score = 1
# ============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 0. Use Beverage-only data
# ------------------------------------------------------------
df_beve = df.loc[df['NewCategory'] == 'Beverages'].copy()

# ------------------------------------------------------------
# 1. Define claim groups and readable labels
# ------------------------------------------------------------
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree',
        'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = {
    'LessCalorie': 'Low/No/Reduced Calorie',
    'NoAddedSugar': 'No Added Sugar',
    'SugarFree': 'Sugar Free',
    'LowSugar': 'Low/Reduced Sugar',
    'Diet': 'Diet/Light',
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessCarb': 'Low/No/Reduced Carb',
    'LessFat': 'Low/No/Reduced Fat',
    'LessTransFat': 'Low/No/Reduced Trans Fat',
    'LessSatFat': 'Low/No/Reduced Saturated Fat',
    'LessChol': 'Low/No/Reduced Cholesterol',
    'LessGlycemic': 'Low/No/Reduced Glycemic',

    'PlusVitamin': 'Vitamin/Mineral Fortified',
    'HighProtein': 'High/Added Protein',
    'AddedCalcium': 'Added Calcium',
    'HighFiber': 'High/Added Fiber',

    'AllNatural': 'All Natural Product',
    'NoArtAdditives': 'No Added/Artificial Additives',
    'NoArtColourings': 'No Added/Artificial Colourings',
    'NoArtFlavourings': 'No Added/Artificial Flavourings',
    'NoArtPreservatives': 'No Added/Artificial Preservatives',
    'NoAdditivesPreservatives': 'No Additives/Preservatives',
    'GMOFree': 'GMO Free',
    'Organic': 'Organic',
    'Wholegrain': 'Whole Grain',

    'BrainNervSystem': 'Brain & Nervous System',
    'ImmuneSystem': 'Immune System',
    'Digestive': 'Digestive',
    'Probiotic': 'Probiotic/Prebiotic',
    'Antioxidant': 'Antioxidant',
    'WgtMuscle': 'Weight & Muscle Gain',
    'Cardiovascular': 'Cardiovascular',
    'BoneSkinHairEyeHealth': 'Bone/Skin/Nails & Hair/Eye Health'
}

claims = [c for grp in claim_groups.values() for c in grp]

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def get_significance_marker(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

def assign_group(claim):
    for group, items in claim_groups.items():
        if claim in items:
            return group
    return 'Other'

# ------------------------------------------------------------
# 3. Compute Beverage-only results
# ------------------------------------------------------------
results = []

for claim in claims:
    if claim not in df_beve.columns:
        continue

    with_claim = df_beve.loc[df_beve[claim] == 1, 'np_score'].dropna()
    without_claim = df_beve.loc[df_beve[claim] == 0, 'np_score'].dropna()

    popularity = df_beve[claim].mean() * 100 if claim in df_beve.columns else np.nan

    mean_with = with_claim.mean() if len(with_claim) > 0 else np.nan
    mean_without = without_claim.mean() if len(without_claim) > 0 else np.nan
    direction = mean_with - mean_without if pd.notna(mean_with) and pd.notna(mean_without) else np.nan

    mw_stat, p_val = np.nan, np.nan
    if len(with_claim) > 0 and len(without_claim) > 0:
        try:
            mw_stat, p_val = mannwhitneyu(
                with_claim,
                without_claim,
                alternative='two-sided'
            )
        except ValueError:
            pass

    results.append({
        'Claim': claim,
        'Claim Label': claim_labels[claim],
        'With Claim': mean_with,
        'Without Claim': mean_without,
        'Mean Difference': direction,
        'Popularity': popularity,
        'MW Statistic': mw_stat,
        'p_value': p_val,
        'N With Claim': len(with_claim),
        'N Without Claim': len(without_claim)
    })

results_df_beve = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Add significance, colors, flags, groups
# ------------------------------------------------------------
results_df_beve['Group'] = results_df_beve['Claim'].apply(assign_group)
results_df_beve['Significance'] = results_df_beve['p_value'].apply(get_significance_marker)
results_df_beve['Direction'] = results_df_beve['With Claim'] - results_df_beve['Without Claim']

deep_palette = sns.color_palette('deep')
blue_deep = deep_palette[0]
orange_deep = deep_palette[1]

results_df_beve['Color'] = results_df_beve.apply(
    lambda row: orange_deep if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else
                blue_deep if pd.notna(row['Direction']) and row['Direction'] < 0 and row['p_value'] < 0.05 else
                'black',
    axis=1
)

results_df_beve['Label'] = results_df_beve.apply(
    lambda row: f"{row['Claim Label']} {row['Significance']}"
    if pd.notna(row['p_value']) and row['p_value'] < 0.05
    else row['Claim Label'],
    axis=1
)

results_df_beve['Flag'] = results_df_beve.apply(
    lambda row: True if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else False,
    axis=1
)

# ------------------------------------------------------------
# 5. Save Beverage table
# ------------------------------------------------------------
table_to_save = results_df_beve[[
    'Group', 'Claim', 'Claim Label', 'With Claim', 'Without Claim',
    'Mean Difference', 'Popularity', 'MW Statistic', 'p_value',
    'Significance', 'N With Claim', 'N Without Claim', 'Flag'
]].copy()

csv_file = os.path.join(SOURCE_DATA_DIR, "Figure4A_Discrepancy_Avg_NP_Score_Beverages_table.xlsx".replace(".xlsx", ".csv"))
excel_file = os.path.join(SOURCE_DATA_DIR, "Figure4A_Discrepancy_Avg_NP_Score_Beverages_table.xlsx")

table_to_save.to_csv(csv_file, index=False)
table_to_save.to_excel(excel_file, index=False)

# ------------------------------------------------------------
# 6. Melt for plotting
# ------------------------------------------------------------
results_melted = results_df_beve.melt(
    id_vars=['Claim', 'Popularity', 'p_value', 'Label', 'Color', 'Flag', 'Group'],
    value_vars=['With Claim', 'Without Claim'],
    var_name='Claim Status',
    value_name='Average NPM Score'
)

label_order = results_df_beve['Label'].tolist()
results_melted['Label'] = pd.Categorical(
    results_melted['Label'],
    categories=label_order,
    ordered=True
)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------
sns.set_theme(style="white")
fig = plt.figure(figsize=(9.2, 9.0))

scatter = sns.scatterplot(
    data=results_melted,
    x='Average NPM Score',
    y='Label',
    hue='Claim Status',
    size='Popularity',
    sizes=(50, 400),
    palette='deep',
    style='Claim Status',
    legend='brief'
)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(0.75)

# Expand right edge so cutoff line does not touch border
x0_tmp, x1_tmp = ax.get_xlim()
right_pad = max(0.12 * (x1_tmp - x0_tmp), 0.25)
ax.set_xlim(x0_tmp, x1_tmp + right_pad)

# color y labels
label_color_map = dict(zip(results_df_beve['Label'], results_df_beve['Color']))
for tick in ax.get_yticklabels():
    txt = tick.get_text()
    if txt in label_color_map:
        tick.set_color(label_color_map[txt])

# ------------------------------------------------------------
# 8. Claim group spans and group labels
# ------------------------------------------------------------
label_to_group = dict(zip(results_df_beve['Label'], results_df_beve['Group']))
spans = []
prev_g, start_i = None, None

for i, lab in enumerate(label_order):
    g = label_to_group.get(lab, 'Other')
    if g != prev_g:
        if prev_g is not None:
            spans.append((start_i, i, prev_g))
        start_i = i
        prev_g = g
if prev_g is not None:
    spans.append((start_i, len(label_order), prev_g))

x0, x1 = ax.get_xlim()

for j, (s, e, g) in enumerate(spans):
    y0 = s - 0.5
    height = e - s
    if j != 0 and j != len(spans) - 1:
        ax.add_patch(
            patches.Rectangle(
                (x0, y0),
                width=(x1 - x0),
                height=height,
                fill=False,
                edgecolor='black',
                linewidth=1.2,
                zorder=1
            )
        )

xspan = x1 - x0
for (s, e, g) in spans:
    ax.text(
        x0 - 0.48 * xspan,
        (s + e - 1) / 2,
        g,
        ha='right',
        va='center',
        fontsize=12,
        rotation=90,
        fontweight='bold',
        color='black'
    )

# ------------------------------------------------------------
# 9. Vertical cutoff line and label
#    cutoff = 1 for beverages
# ------------------------------------------------------------
cutoff_color = '#c44e52'
ax.axvline(
    x=1,
    color=cutoff_color,
    linestyle='--',
    linewidth=2,
    zorder=2
)

x0, x1 = ax.get_xlim()
ymin, ymax = ax.get_ylim()
xspan = x1 - x0
yrange = ymax - ymin

x_cut = 1
x_text = min(x1 - 0.01 * xspan, x_cut + 0.18 * xspan)

y_text = ymin - 0.05 * yrange   

ax.text(
    x_text,
    y_text,
    "NPM threshold     \n(≥1 for “less healthy”)",
    color=cutoff_color,
    fontsize=9,
    ha='right',
    va='center',
    zorder=3
)

# ------------------------------------------------------------
# 10. Formatting
# ------------------------------------------------------------
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.1f}'))

plt.xlabel("Average nutrient profile score", fontsize=12)
plt.ylabel("Claims", fontweight='bold', fontsize=12, labelpad=30)

ax.set_axisbelow(True)
ax.grid(
    True, which='both', axis='both',
    linestyle='--', linewidth=0.8, alpha=0.55, color='gray'
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.16)

# legend
handles, labels = scatter.get_legend_handles_labels()
filtered = [(h, l) for h, l in zip(handles, labels) if l in ['With Claim', 'Without Claim']]
if filtered:
    h, l = zip(*filtered)
    l = ['With claim' if x == 'With Claim' else 'Without claim' for x in l]
    plt.legend(
        h, l,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.13),
        ncol=2,
        fontsize=12,
        frameon=True
    )

# save figure
out = os.path.join(PANEL_DIR, "Figure4A_Discrepancy_Avg_NP_Score_Beverages.png")

for spine in ax.spines.values():
    spine.set_linewidth(0.75)
plt.savefig(out, dpi=1200, bbox_inches="tight")

out_pdf = out.replace(".png", ".pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(os.path.join(PANEL_DIR, "Figure4A.svg"), bbox_inches="tight")

plt.show()

print(f"Saved figure: {out}")
print(f"Saved CSV: {csv_file}")
print(f"Saved Excel: {excel_file}")
print(table_to_save.head())


#### Figure 4B: Component Contribution to Discrepancy for Discrepant Claims of Beverages

In [ ]:
# ============================================================
# Figure 4B (Beverages): minimally revised bar + point + 95% CI
# ============================================================
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu

beve_folder = str(PANEL_DIR / "Beverages")
os.makedirs(beve_folder, exist_ok=True)
df_beve = df.loc[df['NewCategory'] == 'Beverages'].copy()

beve_specs = [
    ('LessSodium', 'Low/No/Reduced Sodium', 'Carbonated Soft Drinks'),
    ('LessSodium', 'Low/No/Reduced Sodium', 'Fruit/Flavoured Still Drinks'),
    ('LessSodium', 'Low/No/Reduced Sodium', 'RTD Coffee & Tea'),
    ('LessSatFat', 'Low/No/Reduced Saturated Fat', 'Fruit/Flavoured Still Drinks')
]

beve_results = []
for claim, claim_label, subcategory in beve_specs:
    subset = df_beve.loc[df_beve['NewSubCategory'] == subcategory]
    result = component_results(subset, claim, f'{claim_label} — {subcategory}')
    result['Claim Label'] = claim_label
    result['Subcategory'] = subcategory
    beve_results.append(result)

beve_source = pd.concat(beve_results, ignore_index=True)
beve_source.to_csv(os.path.join(beve_folder, 'Figure4B_SourceData.csv'), index=False)
beve_source.to_excel(os.path.join(beve_folder, 'Figure4B_SourceData.xlsx'), index=False)

x_bound = 1.0

sns.set_theme(style='white')
fig, axes = plt.subplots(2, 2, figsize=(12.0, 9.0), sharex=True)
for ax, spec, comp_df in zip(axes.ravel(), beve_specs, beve_results):
    claim, claim_label, subcategory = spec
    y = np.arange(len(comp_df))
    colors = [orange_deep if v > 0 else blue_deep if v < 0 else neutral_gray
              for v in comp_df['Signed NPM Contribution']]
    for i, row in comp_df.iterrows():
        c = colors[i]
        ax.errorbar(row['Mean Difference'], i, xerr=row['CI Margin'], fmt='o',
                    color=c, ecolor=c, markersize=5.0, elinewidth=1.35,
                    capsize=3.2, capthick=1.1, zorder=4)
        margin = row['CI Margin'] if pd.notna(row['CI Margin']) else np.nan
        estimate = 0.0 if abs(row['Mean Difference']) < 0.005 else row['Mean Difference']
        value = f"{estimate:.2f} ± {margin:.2f}"
        ax.text(1.04, i, value, transform=ax.get_yaxis_transform(),
                ha='left', va='center', fontsize=6.3, color='#333333', clip_on=False)
    labels = [f"{r['Component Label']} {r['Significance']}".strip()
              for _, r in comp_df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=1.0, zorder=3)
    ax.set_xlim(-x_bound, x_bound)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.grid(True, axis='both', linestyle='--', color='#AAAAAA', linewidth=0.45, alpha=0.75)
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
    ax.set_title(f'{claim_label}\n({subcategory})', fontsize=10)
    ax.set_xlabel('Mean difference in component score\n(with claim − without claim)', fontsize=8)
    ax.text(1.04, 1.015, 'Mean difference\n± 95% CI margin', transform=ax.transAxes,
            ha='left', va='bottom', fontsize=6.1, fontweight='bold')
fig.subplots_adjust(left=0.12, right=0.88, bottom=0.10, top=0.91,
                    wspace=0.78, hspace=0.42)

for ext in ('png', 'pdf', 'svg'):
    path = os.path.join(beve_folder, f'Figure4B_Beverage_Component_Discrepancy_Review.{ext}')
    fig.savefig(path, dpi=600 if ext == 'png' else None, bbox_inches='tight')
plt.show()
print(beve_source[['Panel','Component Label','Mean Difference','CI Lower','CI Upper',
                   'N With Claim','N Without Claim','U Statistic','Exact P Value']])


#### Figure 4: Combine Figure 4A and 4B

In [ ]:
# Final Figure 4: live notebook results, landscape A4, vector output.
figure4a_plot = results_df_beve.copy()
figure4a_source = figure4a_plot[[
    'Group', 'Claim Label', 'With Claim', 'Without Claim',
    'Mean Difference', 'Popularity', 'MW Statistic', 'p_value',
    'Significance', 'N With Claim', 'N Without Claim'
]].copy().rename(columns={
    'Group': 'Claim Group',
    'Claim Label': 'Claim',
    'With Claim': 'Mean NP Score With Claim',
    'Without Claim': 'Mean NP Score Without Claim',
    'Mean Difference': 'Mean Difference (With - Without)',
    'Popularity': 'Claim Prevalence (%)',
    'MW Statistic': 'Mann-Whitney U Statistic',
    'p_value': 'Two-sided P Value',
})
figure4b_source = beve_source[[
    'Claim Label', 'Subcategory', 'Component Label', 'Mean With Claim',
    'Mean Without Claim', 'Mean Difference', 'CI Lower', 'CI Upper',
    'N With Claim', 'N Without Claim', 'U Statistic', 'Exact P Value',
    'Significance'
]].copy().rename(columns={
    'Claim Label': 'Claim',
    'Subcategory': 'Category/Subcategory',
    'Component Label': 'NPM Component',
    'Mean With Claim': 'Mean Component Score With Claim',
    'Mean Without Claim': 'Mean Component Score Without Claim',
    'Mean Difference': 'Mean Difference (With - Without)',
    'CI Lower': '95% CI Lower',
    'CI Upper': '95% CI Upper',
    'U Statistic': 'Mann-Whitney U Statistic',
    'Exact P Value': 'Two-sided P Value',
})
figure4a_source.to_csv(SOURCE_DATA_DIR / "Figure4A_SourceData.csv", index=False)
figure4b_source.to_csv(SOURCE_DATA_DIR / "Figure4B_SourceData.csv", index=False)

fig = plt.figure(figsize=(11.69, 8.27))
outer = fig.add_gridspec(1, 2, width_ratios=[1.42, 1.0],
                         left=0.245, right=0.94, bottom=0.12, top=0.89,
                         wspace=0.42)
ax_a = fig.add_subplot(outer[0])
plot_panel_a(ax_a, figure4a_plot, 1, "Beverages")
panel_heading(fig, 0.075, 0.955, "a",
              "Nutrient profile scores by claim status: market-level patterns")

sub = outer[1].subgridspec(4, 2, width_ratios=[3.1, 1.35],
                           wspace=0.08, hspace=0.62)
panel_order = list(dict.fromkeys(beve_source["Panel"].tolist()))
for i, panel in enumerate(panel_order):
    ax = fig.add_subplot(sub[i, 0])
    block = beve_source.loc[beve_source["Panel"] == panel]
    plot_panel_b(ax, block, (-1.0, 1.0), show_ylabel=False,
                 show_xlabel=(i == len(panel_order) - 1))
    claim_label = block["Claim Label"].iloc[0]
    subcategory = block["Subcategory"].iloc[0]
    ax.set_title(f"{claim_label}\n({subcategory})", fontsize=5.7, pad=4)
    value_ax = fig.add_subplot(sub[i, 1])
    plot_value_column(value_ax, block, show_header=(i == 0))
panel_heading(fig, 0.628, 0.955, "b",
              "Nutrient contributions to discrepancy within categories")
fig.text(0.640, 0.505, "Components of nutrient profile score",
         rotation=90, ha="center", va="center", fontsize=6.0,
         fontweight="bold")

for ext in ("pdf", "svg", "png"):
    kwargs = {"dpi": 300} if ext == "png" else {}
    fig.savefig(MAIN_FIGURE_DIR / f"Figure4_Discrepancy_Beverages.{ext}", **kwargs)
plt.show()


In [ ]:
# ============================================================
# SOURCE-DATA WORKBOOKS AND FINAL VALIDATION
# ============================================================
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

SOURCE_WORKBOOK_DIR = MAIN_OUTPUT_ROOT / "Source_Data_Workbooks"
SOURCE_WORKBOOK_DIR.mkdir(parents=True, exist_ok=True)

def _column_codebook(sheet_name, frame, definitions):
    missing = [column for column in frame.columns if column not in definitions]
    if missing:
        raise KeyError(f"Missing codebook definitions for {sheet_name}: {missing}")
    return pd.DataFrame({
        "Sheet": sheet_name,
        "Column": frame.columns,
        "Definition": [definitions[column] for column in frame.columns],
    })

def _format_source_workbook(path, title, overview_rows, sheets):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        codebook_rows = pd.DataFrame(overview_rows, columns=["Item", "Description"])
        codebook_rows.to_excel(writer, sheet_name="Codebook", index=False, startrow=2)
        startrow = len(codebook_rows) + 5
        combined_codebook = pd.concat([
            _column_codebook(sheet_name, frame, definitions)
            for sheet_name, frame, definitions in sheets
        ], ignore_index=True)
        combined_codebook.to_excel(writer, sheet_name="Codebook", index=False, startrow=startrow)

        for sheet_name, frame, _ in sheets:
            frame.to_excel(writer, sheet_name=sheet_name, index=False)

        wb = writer.book
        dark_blue = "245681"
        header_blue = "2F75B5"
        light_border = Side(style="thin", color="D9E2F3")

        codebook = wb["Codebook"]
        codebook["A1"] = title
        codebook.merge_cells("A1:C1")
        codebook["A1"].fill = PatternFill("solid", fgColor=dark_blue)
        codebook["A1"].font = Font(name="Arial", size=14, bold=True, color="FFFFFF")
        codebook["A1"].alignment = Alignment(vertical="center")
        codebook.row_dimensions[1].height = 24
        for row_number in (3, startrow + 1):
            for cell in codebook[row_number]:
                cell.fill = PatternFill("solid", fgColor=header_blue)
                cell.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
        codebook.column_dimensions["A"].width = 20
        codebook.column_dimensions["B"].width = 38
        codebook.column_dimensions["C"].width = 82
        for row in codebook.iter_rows():
            for cell in row:
                cell.font = cell.font.copy(name="Arial")
                cell.alignment = Alignment(vertical="top", wrap_text=True)
                cell.border = Border(bottom=light_border)
        codebook.freeze_panes = f"A{startrow + 2}"
        codebook.sheet_view.showGridLines = False

        for sheet_name, frame, _ in sheets:
            ws = wb[sheet_name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            ws.sheet_view.showGridLines = False
            for cell in ws[1]:
                cell.fill = PatternFill("solid", fgColor=header_blue)
                cell.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
                cell.alignment = Alignment(vertical="center", wrap_text=True)
            ws.row_dimensions[1].height = 34
            for column_number, column_name in enumerate(frame.columns, start=1):
                values = [str(column_name)] + [str(v) for v in frame[column_name].dropna().head(200)]
                width = min(max(len(v) for v in values) + 2, 38)
                ws.column_dimensions[get_column_letter(column_number)].width = max(width, 12)
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.font = Font(name="Arial", size=9)
                    cell.alignment = Alignment(vertical="top")
                    if isinstance(cell.value, float):
                        cell.number_format = "0.000"

def _assert_source_data():
    assert np.allclose(
        figure3a_source["Mean NP Score With Claim"] - figure3a_source["Mean NP Score Without Claim"],
        figure3a_source["Mean Difference (With - Without)"], equal_nan=True)
    assert np.allclose(
        figure4a_source["Mean NP Score With Claim"] - figure4a_source["Mean NP Score Without Claim"],
        figure4a_source["Mean Difference (With - Without)"], equal_nan=True)
    for frame in (figure3b_source, figure4b_source):
        assert (frame["95% CI Lower"] <= frame["95% CI Upper"]).all()
        assert (frame["N With Claim"] > 0).all() and (frame["N Without Claim"] > 0).all()
    for frame in (figure1_source, figure2_source, figure3a_source,
                  figure3b_source, figure4a_source, figure4b_source):
        assert not any(str(column).lower() == "flag" for column in frame.columns)

definitions_figure1 = {
    "Year": "Product launch year.",
    "Series": "Displayed bar or line series.",
    "Share of products (%)": "Percentage of products represented by the displayed series in the specified year.",
}
definitions_figure2 = {
    "Claim": "Displayed on-package claim.",
    "Product Category": "Product category represented by the heatmap column.",
    "Estimated Share (%)": "Estimated percentage of products in the category carrying the claim.",
    "95% CI Margin (percentage points)": "Displayed half-width of the 95% confidence interval.",
    "95% CI Lower (%)": "Estimated share minus the displayed confidence-interval margin.",
    "95% CI Upper (%)": "Estimated share plus the displayed confidence-interval margin.",
}
definitions_panel_a = {
    "Claim Group": "Claim classification used to organize Panel a.",
    "Claim": "Displayed on-package claim.",
    "Mean NP Score With Claim": "Arithmetic mean nutrient profile score among products carrying the claim.",
    "Mean NP Score Without Claim": "Arithmetic mean nutrient profile score among products not carrying the claim.",
    "Mean Difference (With - Without)": "Mean NP score with claim minus mean NP score without claim.",
    "Claim Prevalence (%)": "Percentage of products carrying the claim; encoded by marker size.",
    "Mann-Whitney U Statistic": "Test statistic for the two-sided Wilcoxon rank-sum/Mann-Whitney U comparison.",
    "Two-sided P Value": "Unadjusted two-sided p value for the claim-status comparison.",
    "Significance": "Asterisk notation displayed beside the claim label.",
    "N With Claim": "Number of products carrying the claim.",
    "N Without Claim": "Number of products not carrying the claim.",
}
definitions_panel_b = {
    "Claim": "Displayed on-package claim.",
    "Category/Subcategory": "Within-category product comparison represented by the forest-plot subpanel.",
    "NPM Component": "Displayed component of the nutrient profile score.",
    "Mean Component Score With Claim": "Arithmetic mean component score among products carrying the claim.",
    "Mean Component Score Without Claim": "Arithmetic mean component score among products not carrying the claim.",
    "Mean Difference (With - Without)": "Mean component score with claim minus mean component score without claim.",
    "95% CI Lower": "Lower bound of the 95% Wald confidence interval for the mean difference.",
    "95% CI Upper": "Upper bound of the 95% Wald confidence interval for the mean difference.",
    "N With Claim": "Number of products carrying the claim.",
    "N Without Claim": "Number of products not carrying the claim.",
    "Mann-Whitney U Statistic": "Test statistic for the two-sided Wilcoxon rank-sum/Mann-Whitney U comparison.",
    "Two-sided P Value": "Unadjusted two-sided p value used for the asterisk notation.",
    "Significance": "Asterisk notation displayed beside the component label.",
}

_assert_source_data()

_format_source_workbook(
    SOURCE_WORKBOOK_DIR / "Figure1_SourceData.xlsx", "Figure 1 source data",
    [("Figure", "Figure 1"),
     ("Contents", "Annual shares of products classified as less healthy under the NPM and carrying each claim group, 2015–2024."),
     ("Methods", "Values correspond to the bars, lines, markers, and numeric labels displayed in Figure 1."),
     ("Rounding", "Workbook cells retain numeric values; display formatting may show fewer decimal places.")],
    [("Figure 1", figure1_source, definitions_figure1)],
)
_format_source_workbook(
    SOURCE_WORKBOOK_DIR / "Figure2_SourceData.xlsx", "Figure 2 source data",
    [("Figure", "Figure 2"),
     ("Contents", "Claim prevalence estimates and 95% confidence-interval margins by product category."),
     ("Methods", "The heatmap displays each estimate as estimate ± 95% CI margin; intervals use the normal approximation."),
     ("Rounding", "Lower and upper bounds are reconstructed from the displayed rounded estimate and margin.")],
    [("Figure 2", figure2_source, definitions_figure2)],
)
_format_source_workbook(
    SOURCE_WORKBOOK_DIR / "Figure3_SourceData.xlsx", "Figure 3 source data",
    [("Figure", "Figure 3"),
     ("Contents", "Panel 3a contains market-wide food comparisons; Panel 3b contains within-category NPM-component comparisons."),
     ("Methods", "Asterisks are based on unadjusted two-sided Wilcoxon rank-sum/Mann–Whitney U tests. Panel 3b whiskers are 95% Wald confidence intervals."),
     ("NPM interpretation", "Higher scores indicate less favorable nutritional quality; food products with scores ≥4 are classified as less healthy."),
     ("Direction", "Positive mean differences indicate higher scores among products with the claim."),
     ("Significance", "* p<0.05; ** p<0.01; *** p<0.001; blank indicates p≥0.05.")],
    [("Figure 3a", figure3a_source, definitions_panel_a),
     ("Figure 3b", figure3b_source, definitions_panel_b)],
)
_format_source_workbook(
    SOURCE_WORKBOOK_DIR / "Figure4_SourceData.xlsx", "Figure 4 source data",
    [("Figure", "Figure 4"),
     ("Contents", "Panel 4a contains market-wide beverage comparisons; Panel 4b contains within-category NPM-component comparisons."),
     ("Methods", "Asterisks are based on unadjusted two-sided Wilcoxon rank-sum/Mann–Whitney U tests. Panel 4b whiskers are 95% Wald confidence intervals."),
     ("NPM interpretation", "Higher scores indicate less favorable nutritional quality; beverages with scores ≥1 are classified as less healthy."),
     ("Direction", "Positive mean differences indicate higher scores among products with the claim."),
     ("Significance", "* p<0.05; ** p<0.01; *** p<0.001; blank indicates p≥0.05.")],
    [("Figure 4a", figure4a_source, definitions_panel_a),
     ("Figure 4b", figure4b_source, definitions_panel_b)],
)

print("All Figures 1–4 and source-data files were generated successfully.")
print(f"Figures: {MAIN_FIGURE_DIR}")
print(f"CSV source data: {SOURCE_DATA_DIR}")
print(f"Submission workbooks: {SOURCE_WORKBOOK_DIR}")
